In [2]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('market.csv')
print(df.head())
print(df.shape)

item_counts = df.sum().sort_values(ascending=False)
print(item_counts.head(10))

min_support_rate = 0.05
min_support_count = int(df.shape[0] * min_support_rate)
print(f"Min Support Count: {min_support_count}")

transactions = [set(df.columns[row == 1]) for _, row in df.iterrows()]

  Bread;Honey;Bacon;Toothpaste;Banana;Apple;Hazelnut;Cheese;Meat;Carrot;Cucumber;Onion;Milk;Butter;ShavingFoam;Salt;Flour;HeavyCream;Egg;Olive;Shampoo;Sugar
0        1;0;1;0;1;1;1;0;0;1;0;0;0;0;0;0;0;1;1;0;0;1                                                                                                        
1        1;1;1;0;1;1;1;0;0;0;1;0;1;1;0;0;1;0;0;1;1;0                                                                                                        
2        0;1;1;1;1;1;1;1;1;0;1;1;1;0;1;1;1;1;1;0;0;1                                                                                                        
3        1;1;0;1;0;1;0;0;0;0;1;1;1;0;0;0;1;0;1;1;1;0                                                                                                        
4        0;1;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0                                                                                                        
(464, 1)
Bread;Honey;Bacon;Toothpaste;Banana;Apple;Hazelnu

In [3]:
from itertools import combinations

def get_frequent_itemsets(transactions, min_support_count):
    item_counts = {}
    for transaction in transactions:
        for item in transaction:
            item_counts[item] = item_counts.get(item, 0) + 1
    
    current_frequent = {frozenset([item]): count for item, count in item_counts.items() if count >= min_support_count}
    all_frequent = dict(current_frequent)
    k = 2
    
    while current_frequent:
        items = list(current_frequent.keys())
        candidates = set()
        
        for i in range(len(items)):
            for j in range(i + 1, len(items)):
                union = items[i] | items[j]
                if len(union) == k:
                    subsets = combinations(union, k - 1)
                    if all(frozenset(s) in current_frequent for s in subsets):
                        candidates.add(union)
        
        if not candidates:
            break
            
        counts = {c: 0 for c in candidates}
        for transaction in transactions:
            for candidate in candidates:
                if candidate.issubset(transaction):
                    counts[candidate] += 1
        
        current_frequent = {c: count for c, count in counts.items() if count >= min_support_count}
        all_frequent.update(current_frequent)
        k += 1
        
    return all_frequent

frequent_itemsets = get_frequent_itemsets(transactions, min_support_count)
print(f"Znaleziono {len(frequent_itemsets)} częstych zbiorów.")

Znaleziono 0 częstych zbiorów.


In [4]:
def generate_rules(frequent_itemsets, min_confidence, total_transactions):
    rules = []
    for itemset, support_count in frequent_itemsets.items():
        if len(itemset) < 2:
            continue
        
        for i in range(1, len(itemset)):
            for antecedent in combinations(itemset, i):
                antecedent = frozenset(antecedent)
                consequent = itemset - antecedent
                
                support_a = frequent_itemsets[antecedent]
                confidence = support_count / support_a
                
                if confidence >= min_confidence:
                    support_rule = support_count / total_transactions
                    support_b = frequent_itemsets[consequent]
                    
                    lift = confidence / (support_b / total_transactions)
                    leverage = support_rule - (support_a / total_transactions) * (support_b / total_transactions)
                    
                    rules.append({
                        'antecedent': set(antecedent),
                        'consequent': set(consequent),
                        'support': support_rule,
                        'confidence': confidence,
                        'lift': lift,
                        'leverage': leverage
                    })
    return rules

min_confidence = 0.4
rules = generate_rules(frequent_itemsets, min_confidence, len(transactions))
print(f"Wygenerowano {len(rules)} reguł.")

Wygenerowano 0 reguł.


In [5]:
rules_df = pd.DataFrame(rules)

print("Top 5 wg Lift:")
print(rules_df.sort_values(by='lift', ascending=False).head(5))

print("\nTop 5 wg Leverage:")
print(rules_df.sort_values(by='leverage', ascending=False).head(5))

Top 5 wg Lift:


KeyError: 'lift'

In [ ]:
n_target = 1000000
scale_factor = n_target // len(transactions)
large_transactions = transactions * scale_factor

print(f"Liczba transakcji: {len(large_transactions)}")

min_support_large = int(len(large_transactions) * 0.05)
frequent_large = get_frequent_itemsets(large_transactions, min_support_large)

print(f"Liczba zbiorów: {len(frequent_large)}")

In [ ]:
import numpy as np

n_new_items = 1000
n_rows = 2000 

new_data = np.random.choice([0, 1], size=(n_rows, n_new_items), p=[0.99, 0.01])
new_columns = [f"Item_{i}" for i in range(n_new_items)]
df_new_items = pd.DataFrame(new_data, columns=new_columns)

df_base = df.sample(n_rows, replace=True).reset_index(drop=True)
df_combined = pd.concat([df_base, df_new_items], axis=1)

combined_transactions = [set(df_combined.columns[row == 1]) for _, row in df_combined.iterrows()]
print(f"Wymiary: {len(combined_transactions)} transakcji, {len(df_combined.columns)} produktów")

start_time = time.time()

min_support_items = int(len(combined_transactions) * 0.05) 
frequent_items = get_frequent_itemsets(combined_transactions, min_support_items)

end_time = time.time()
print(f"Czas obliczeń: {end_time - start_time:.4f} s")
print(f"Liczba zbiorów: {len(frequent_items)}")